# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Explore the schema to discover how the data is organized and referenced.

Entities such as record sets, fields, and columns must be referenced by their `@id`. Any downstream data manipulation should use these IDs for consistency.

In [ ]:
# Get the available record sets and their @ids
print("Available record sets and fields:")
for record_set in metadata.record_sets:
    print(f"- Record set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '')}")
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields @ids:")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    - {field_id}")
    print("  Columns:")
    if 'column' in record_set:
        columns = record_set['column']
        if isinstance(columns, dict):
            columns = [columns]
        for column in columns:
            column_id = column['@id'] if isinstance(column, dict) else column
            print(f"    - {column_id}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. All references to record sets, fields, and columns are by their `@id` values from the overview above.

In [ ]:
# Specify record sets to extract, by @id
# Replace with the actual @ids from the overview above if they differ.
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set {record_set_id}.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# List columns for each loaded dataframe
for record_set_id, df in dataframes.items():
    print(f"\nRecord set {record_set_id} columns:")
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

For this section, we'll select a numeric field (such as `cr:coverage_percentage` or `cr:molecular_weight`) and a grouping field, based on the available fields in your record set. These fields must be referenced by their `@id`.

In [ ]:
# Choose a main record set for EDA (replace with desired @id from the list above)
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Inspect and select a numeric field by @id (e.g., 'cr:coverage_percentage', 'cr:molecular_weight', etc.)
# If uncertain, print column names and inspect available options
print("Columns for main record set:", df.columns.tolist())
numeric_field = None
candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    # try to convert eligible columns to numeric
    for col in df.columns:
        try:
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notnull().mean() > 0.5:
                numeric_field = col
                df[col] = converted
                break
        except:
            continue

if numeric_field is None:
    raise ValueError("No numeric field found for EDA.")
print(f"Selected numeric field for analysis: {numeric_field}")

# Example threshold filtering
threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt grouping by a categorical field, e.g. 'cr:description' or other text @id
candidate_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
group_field = candidate_group_fields[0] if candidate_group_fields else None
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
    print(f"Grouped data by {group_field}, mean {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot a histogram of the selected numeric field and, if possible, a boxplot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If a group_field exists, create a boxplot by that field
if group_field is not None and df[group_field].nunique() < 20:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} grouped by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, you have:
- Loaded a complex biomedical data package described by a Croissant schema
- Explored record sets, fields, and their unique `@id` references
- Extracted and inspected data using those `@id`s
- Performed basic filtering, normalization, and grouping
- Plotted distribution(s) to visualize patterns

**Key observations will depend on your selection of fields and record sets.**

You can further analyze or process the dataset by referencing fields exclusively by their `@id` following the Croissant data contract.